In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.style as style
from statsmodels.tsa.api import seasonal_decompose
import datetime
import time
#import pandas.util.testing as tm
import matplotlib.pyplot as mp
style.use('ggplot')
from pandas.tseries.offsets import MonthEnd
import datetime as dt
from sklearn.linear_model import LinearRegression

# **Funções Utilizadas**

In [ ]:
# Funções

# fix_size = resize_dataframe, resample_df_escala2=resample_dataframe_by_scale
def resize_dataframe(input_df):
    """
    Resize the DataFrame by adding rows at the beginning and the end based on the unique 'CodProduto'.
    """
    # Define start and end dates
    start_date = pd.Timestamp('2023-10-01')
    end_date = pd.Timestamp('2023-10-31')

    # Count the number of unique products
    unique_products = input_df['CodProduto'].unique()
    num_unique_products = len(unique_products)

    # Create initial DataFrame (df_initial) with rows set to the start date
    df_initial = input_df.tail(num_unique_products).reset_index(drop=True)
    df_initial['DtEmissao'] = start_date
    df_initial['CodProduto'] = unique_products
    df_initial[['QuantdadeItem', 'PrecoItem', 'TotalItem']] = 0.0
    df_initial[['TpMovi', 'serie', 'documento', 'IdFilial']] = ['FAKE', 999, 0, 999]

    # Create end DataFrame (df_end) with rows set to the end date
    df_end = input_df.tail(num_unique_products).reset_index(drop=True)
    df_end['DtEmissao'] = end_date
    df_end['CodProduto'] = unique_products
    df_end[['QuantdadeItem', 'PrecoItem', 'TotalItem']] = 0.0
    df_end[['TpMovi', 'serie', 'documento', 'IdFilial']] = ['FAKE', 999, 0, 999]

    # Append the initial and end DataFrames to the original DataFrame
    resized_df = pd.concat([input_df, df_initial, df_end], ignore_index=True)

    # Sort by date and reset index
    resized_df = resized_df.sort_values(by='DtEmissao').reset_index(drop=True)

    return resized_df

from tqdm import tqdm

def process(dataframe, produto):
    return dataframe[dataframe['CodProduto'] == produto]

def ind_prod(lst, produto):
    return lst.index(produto)

def resample_df(dataframe, temporal):
    return dataframe.resample(temporal, on='DtEmissao', closed='right', label='right').sum()

def reset_index(dataframe):
    return dataframe.reset_index()


# Resample não mais utilizada (datas nas linhas)
def resample_df_escala(dataframe, escala):
    # Resample para escala = n : n meses

    scale = escala
    df = dataframe

    part2 = df[len(df)%scale:]
    str_columns = [col for col in part2.columns if any(part2[col].apply(lambda x: isinstance(x, str)))]
    print("Colunas que contêm elementos do tipo 'str':", str_columns)
    df_aux = (part2.resample('{}M'.format(scale), on='DtEmissao', label='right', closed='left').sum())
    return df_aux

# Resample que funciona com as transpostas (datas nas colunas)
def resample_df_escala2(dataframe, escala):
    dataframe.columns = pd.to_datetime(dataframe.columns, errors='coerce')
    dataframe = dataframe.loc[:, dataframe.columns.dropna()]
    display(dataframe)
    print('tipo de elemento da coluna', type(dataframe.columns[0]))
    # Resample para escala = n : n meses
    scale = escala
    df = dataframe
    part2 = df.iloc[:, len(df.columns)%scale:]
    display(part2)
    str_columns = [col for col in part2.columns if any(part2[col].apply(lambda x: isinstance(x, str)))]
    print("Colunas que contêm elementos do tipo 'str':", str_columns)
    part2.info()
    #display(part2.iloc[:, 57])
    #display(part2.iloc[:, 58])
    #display(part2.iloc[:, 65])
    df_filtered = dataframe[dataframe == '0 days'].dropna(how='all').dropna(axis=1, how='all')
    print(df_filtered)
    part2 = part2.drop(['DtEmissao'], axis=0)
    part2 = part2.astype(float)
    df_aux = (part2.resample('{}M'.format(scale), axis='columns', label='left', closed='left').sum())
    return df_aux

def cria_df_historico(list_dataframes, list_produtos):
    df_aux = list_dataframes[0].iloc[:, 0:1]
    l_str = list(map(str, list_produtos))
    blot = [list_dataframes[idx].QuantdadeItem for idx in tqdm(range(len(list_produtos)))]
    return pd.concat([df_aux] + blot, axis=1).rename(columns={'DtEmissao': 'DtEmissao', **dict(zip(range(len(l_str)), l_str))})

def eliminate_doubles(dataframe_compras, lista_fornecedor):
    return dataframe_compras[dataframe_compras['CodProduto'].isin(lista_fornecedor)].drop_duplicates().reset_index(drop=True)

# Intersecção e diferença
def inter_dif(lst1, lst2):
    intersection = set(lst1).intersection(lst2)
    difference1 = set(lst1).difference(set(lst2))
    difference2 = set(lst2).difference(set(lst1))
    print('total primeira: ', len(lst1))
    print('total segunda: ', len(lst2))
    print('intersecção: ', len(intersection))
    print('primeira - segunda: ', len(difference1))
    print('segunda - primeira', len(difference2))
    return list(intersection), list(difference1), list(difference2)

def atualizar2(dataframe_completo, dataframe_mes):
    n_meses = len(dataframe_completo.columns)
    print(n_meses)
    print(len(dataframe_mes.columns))

    intersection, difference1, difference2 = inter_dif(dataframe_completo.index, dataframe_mes.index)

    # Create DataFrame for the items to complete
    df_a_completar = pd.DataFrame(0.0, index=difference1, columns=dataframe_mes.columns)

    # Join the intersected data
    result = pd.concat([dataframe_mes.loc[intersection], df_a_completar])

    # Update the column names for the result DataFrame
    last_col_date = datetime.datetime.strptime(dataframe_completo.columns[-1], "%Y-%m-%d")
    result.columns = [last_col_date + MonthEnd(1)]

    # Join with the complete DataFrame
    print('df completo')
    display(dataframe_completo)
    print('result')
    display(result)
    horizontal = pd.concat([dataframe_completo, result], axis=1, join='inner')

    # Handle new entries
    df_zeros = pd.DataFrame(0.0, index=difference2, columns=dataframe_completo.columns)
    df_novos = dataframe_mes.loc[difference2]
    df_novos.columns = result.columns
    df_novos = pd.concat([df_zeros, df_novos], axis=1, join='inner')
    print('horizontal')
    display(horizontal)
    print('df_novos')
    display(df_novos)
    df_finalM = pd.concat([horizontal, df_novos], axis=0, join="inner")

    df_finalM.iloc[-1] = pd.to_numeric(df_finalM.iloc[-1])

    df_final2M = resample_df_escala2(df_finalM, 2)  # Assuming resample_df_escala2 is defined elsewhere
    df_final3M = resample_df_escala2(df_finalM, 3)  # Assuming resample_df_escala2 is defined elsewhere

    return df_finalM, df_final2M, df_final3M

def linear_prediction(dataframe, horizon):
    df = dataframe
    h = horizon
    df2 = df.iloc[:, 1:]
    previsoes = [list(previsao_linear(df, produto, h)) for produto in df2.columns]

    return previsoes

def previsao_linear(dataframe, produto, horizon):
    df = dataframe
    h = horizon
    print(produto)
    t = df.DtEmissao
    #print('t antes')
    #print(t)
    for i in list(range(h)):
        t = t._append(t[-1:])
        t[-1:] = t[-1:] + MonthEnd(1)

    #print('t depois')
    #print(t)
    t_mod = t.map(dt.datetime.toordinal)
    #print('t_mod')
    #print(t_mod)
    t_pred = t_mod[-h:]
    t_pred = [t_pred]
    #print('t_pred')
    #print(t_pred)

    # Training data
    X = t_mod[:-h]  # features
    #print('X')
    #print(X)
    #Xm = [[element] for element in X]
    Xm = X.values.reshape(-1, 1)
    #print('len(Xm)', len(Xm))
    y = df.loc[:, produto]  # target
    #print('Xm')
    #print(Xm)
    #print('y')
    #print(y)

    # Train the model
    model = LinearRegression()
    model.fit(Xm, y)

    #print('passou')
    # Store the fitted values as a time series with the same time index as
    # the training data
    #y_pred = pd.Series(model.predict(X), index=X.index)
    y_pred = model.predict(t_pred)

    return y_pred

# **SVM**

In [ ]:
from math import floor

lagnumber = 12

class Product:
    def __init__(self, raw_data):
        self.raw_data = raw_data
        self.id = raw_data["CodProduto"].iloc[0]
        self.raw_sales = raw_data[["DtEmissao", "QuantdadeItem"]]

    def make_returns(self):
        change = self.raw_sales['QuantdadeItem'].div(self.raw_sales["QuantdadeItem"].shift(1)) - 1
        self.data = {'DtEmissao':self.raw_sales['DtEmissao'],'QuantdadeItem':self.raw_sales['QuantdadeItem'],
                'Relative change over the next one month':change.shift(-1),
                'Lag 0 months':change}

                #, 'Lag 1 months':change.shift(1),
                #'Lag 2 months':change.shift(2), 'Lag 3 months':change.shift(3),
                #'Lag 4 months':change.shift(4), 'Lag 5 months':change.shift(5),
                #'Lag 6 months':change.shift(6), 'Lag 7 months':change.shift(7),
                #'Lag 8 months':change.shift(8), 'Lag 9 months':change.shift(9),
                #'Lag 10 months':change.shift(10), 'Lag 11 months':change.shift(11),
                #'Lag 12 months':change.shift(12)}
        i = 0
        while i <= lagnumber-1:
          self.data.update({'Lag '+str(i)+' months':change.shift(i) })
          i = i + 1

        self.data = pd.DataFrame(data=self.data)

    def train_val(self):
        #print("Processing train and val data\n")
        data = self.data.iloc[lagnumber::,:]
        self.n = len(self.data) - lagnumber # length of dataframe // lagnumber = Number of lags (removed because lagmatrix shifts)
        # TODO: Must check if there are any gaps between months!! For now we are choosing 'regular' products (prod = 35190001640)
        self.n_train = floor(0.8*self.n)
        self.n_val = self.n-self.n_train
        self.data_train = data.iloc[0:self.n_train]
        self.data_val = data.iloc[self.n_train::]
        self.data_val = self.data_val.iloc[:-1,:]
        #print("Done!")

In [ ]:
# -*- coding: utf-8 -*-
__all__ = ["LSSVMRegression",]
"""
Created on Tue May 19 09:27:21 2020
An LS-SVM regression class following the sk-learn API.
_                         _   _   _    _  _
| 0          1^T_N         |  | b  |   | 0 |
|                          |  |    | = |   |
| 1_N  Omega+gamma^-1 I_N  |  | a  |   | Y |
|_                        _|  |_  _|   |_ _|
Omega= Kernel K(x_i,x_j)
gamma= hyper-parameter (is a ratio z/µ with z the sum squared error and µ the
                        amount of regularization)
1_N = vector (1,1,1,..,1)
I_N = NxN unity matrix
@author: Dr. Dr. Danny E. P. Vanpoucke
@web   : https://dannyvanpoucke.be
"""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin


class LSSVMRegression(BaseEstimator, RegressorMixin):
    """
    An Least Squared Support Vector Machine (LS-SVM) regression class, build
    on the BaseEstimator and RegressorMixin base classes of sklearn.
    (Let's hope furture upgrades of python sk-learn just doesn't break this...
    consider this a python feature)
    Attributes:
        - gamma : the hyper-parameter (float)
        - kernel: the kernel used     (string)
        - kernel_: the actual kernel function
        - x : the data on which the LSSVM is trained (call it support vectors)
        - y : the targets for the training data
        - coef_ : coefficents of the support vectors
        - intercept_ : intercept term
    """
    def __init__(self, gamma: float = 1.0, kernel: str = None, c: float = 1.0,
                 d: float = 2, sigma: float = 1.0):
        """
        Create a new regressor
        Parameters:
            - gamma: floating point value for the hyper-parameter gamma, DEFAULT=1.0
            - kernel: string indicating the kernel: {'linear','poly','rbf'}, DEFAULT='rbf'
            - the kernel parameters
                    * linear: none
                    * poly:
                        + c: scaling constant, DEFAULT=1.0
                        + d: polynomial power, DEFAULT=2
                    * rbf:
                        + sigma: scaling constant, DEFAULT=1.0
        """
        self.gamma = gamma
        self.c = c
        self.d = d
        self.sigma = sigma
        if kernel is None:
            self.kernel = 'rbf'
        else:
            self.kernel = kernel

        params = dict()
        if kernel == 'poly':
            params['c'] = c
            params['d'] = d
        elif kernel == 'rbf':
            params['sigma'] = sigma

        self.kernel_ = LSSVMRegression.__set_kernel(self.kernel, **params)

        #model parameters
        self.x = None
        self.y = None
        self.coef_ = None
        self.intercept_ = None

    def get_params(self, deep=True):
        """
            The get_params functionality provides the parameters of the LSSVMRegression class.
            These exclude the modelparameters.
        """
        return {"c": self.c, "d": self.d, "gamma": self.gamma,
                "kernel": self.kernel, "sigma":self.sigma}

    def set_params(self, **parameters):
        """
            Set the parameters of the class. Important note: This should do
            anything that is done to relevant parameters in __init__ as
            sklearn's GridSearchCV uses this instead of init.
            More info:  https://scikit-learn.org/stable/developers/develop.html
        """
        #print("SETTING PARAMETERS IN LSSVM:",parameters.items())

        for parameter, value in parameters.items():
            #setattr should do the trick for gamma,c,d,sigma and kernel
            setattr(self, parameter, value)
        #now also update the actual kernel
        params = dict()
        if self.kernel == 'poly':
            params['c'] = self.c
            params['d'] = self.d
        elif self.kernel == 'rbf':
            params['sigma'] = self.sigma
        self.kernel_ = LSSVMRegression.__set_kernel(self.kernel, **params)

        return self

    def set_attributes(self, **parameters):
        """
            Manually set the attributes of the model. This should generally
            not be done, except when testing some specific behaviour, or
            creating an averaged model.
            Parameters are provided as a dictionary.
                - 'intercept_' : float intercept
                - 'coef_'      : float array of coefficients
                - 'support_'   : array of support vectors, in the same order sorted
                                 as the coefficients
        """
        #not the most efficient way of doing it...but sufficient for the time being
        for param, value in parameters.items():
            if param == 'intercept_':
                self.intercept_ = value
            elif param == 'coef_':
                self.coef_ = value
            elif param == 'support_':
                self.x = value

    @staticmethod
    def __set_kernel(name: str, **params):
        """
            Internal static function to set the kernel function.
            NOTE: The second "vector" xj will be the one which generally
                  contains an array of possible vectors, while xi should be a single
                  vector. Therefore, the numpy dot-product requires xj to
                  be transposed.
            The kernel returns either a scalar or a numpy nd-array of
            rank 1 (i.e. a vector), if it returns something else the result
            is wrong if xi is an array.
        """
        def linear(xi, xj):
            """
               v*v=scal (dot-product OK)
               v*m=v    (dot-product OK)
               m*m=m    (matmul for 2Dx2D, ok with dot-product)
            """
            return np.dot(xi, xj.T)

        def poly(xi, xj, c=params.get('c', 1.0), d=params.get('d', 2)):
            """
                Polynomial kernel ={1+ (xi*xj^T)/c }^d
                Parameters:
                    - c: scaling constant, DEFAULT=1.0
                    - d: polynomial power, DEFAULT=2
                    - xi and xj are numpy nd-arrays
                (cf: https://en.wikipedia.org/wiki/Least-squares_support-vector_machine )
                works on same as linear
            """
            return ((np.dot(xi, xj.T))/c  + 1)**d

        def rbf(xi, xj, sigma=params.get('sigma', 1.0)):
            """
            Radial Basis Function kernel= exp(- ||xj-xi||² / (2*sigma²))
            In this formulation, the rbf is also known as the Gaussian kernel of variance sigma²
            As the Euclidean distance is strict positive, the results of this kernel
            are in the range [0..1] (x € [+infty..0])
            Parameters:
                - sigma: scaling constant, DEFAULT=1.0
                - xi and xj are numpy nd-arrays
            (cf: https://en.wikipedia.org/wiki/Least-squares_support-vector_machine )
            Possible combinations of xi and xj:
                vect & vect   -> scalar
                vect & array  -> vect
                array & array -> array => this one requires a pair distance...
                                    which can not be done with matmul and dot
                The vectors are the rows of the arrays (Arr[0,:]=first vect)
                The squared distance between vectors= sqr(sqrt( sum_i(vi-wi)² ))
                --> sqr & sqrt cancel
                --> you could use a dot-product operator for vectors...but this
                seems to fail for nd-arrays.
            For vectors:
                ||x-y||²=sum_i(x_i-y_i)²=sum_i(x²_i+y²_i-2x_iy_i)
                --> all products between vectors can be done via np.dot: takes the squares & sum
            For vector x and array of vectors y:
                --> x²_i : these are vectors: dot gives a scalar
                --> y²_i : this should be a list of scalars, one per vector.
                            => np.dot gives a 2d array
                            => so   1) square manually (squares each element)
                                    2) sum over every row (axis=1...but only in case we
                                                           have a 2D array)
                --> x_iy_i : this should also be a list of scalars. np.dot does the trick,
                            and even gives the same result if matrix and vector are exchanged
            for array of vectors x and array of vectors y:
                --> either loop over vectors of x, and for each do the above
                --> or use cdist which calculates the pairwise distance and use that in the exp
            """
            from scipy.spatial.distance import cdist

           # print('LS_SVM DEBUG: Sigma=',sigma,'  type=',type(sigma) )
           # print('              xi   =',xi,'  type=',type(xi))
           # print('              xj   =',xj,'  type=',type(xj))


            if (xi.ndim == 2 and xi.ndim == xj.ndim): # both are 2D matrices
                return np.exp(-(cdist(xi, xj, metric='sqeuclidean'))/(2*(sigma**2)))
            elif ((xi.ndim < 2) and (xj.ndim < 3)):
                ax = len(xj.shape)-1 #compensate for python zero-base
                return np.exp(-(np.dot(xi, xi) + (xj**2).sum(axis=ax)
                                - 2*np.dot(xi, xj.T))/(2*(sigma**2)))
            else:
                message = "The rbf kernel is not suited for arrays with rank >2"
                raise Exception(message)

        kernels = {'linear': linear, 'poly': poly, 'rbf': rbf}
        if kernels.get(name) is not None:
            return kernels[name]
        else: #unknown kernel: crash and burn?
            message = "Kernel "+name+" is not implemented. Please choose from : "
            message += str(list(kernels.keys())).strip('[]')
            raise KeyError(message)

    def __OptimizeParams(self):
        """
        Solve the matrix operation to get the coefficients.
        --> equation 3.5 and 3.6 of the book by Suykens
        ==> that is for classification, for regression slightly different cf Dilmen paper 2017
        self.y: 1D array
        self.X: 2D array (with rows the vectors: X[0,:] first vector)
        Set the class parameters:
            - self.intercept_ : intercept
            - self.coef_      : coefficients
        """
        #eq 3.6: Omega_kl = y_ky_lK(x_k,x_l)
        # !! note that the product of a vector and the transposed is a dot-product
        #    and we need an outer product
        #For classification and Regression, the matrices are slightly different...
        # (why? except for what came out of solving equations?
        # Dilmen et al, IFAC PapersOnline 50(1), 8642-8647 (2017))

        # Classification
        # Omega = np.multiply( np.multiply.outer(y,y), self.kernel_(X,X) ) # correct version
        #A_dag = np.linalg.pinv(np.block([
        #    [0,                           y.T                   ],
        #    [y,   Omega + self.gamma**-1 * np.eye(len(y_values))]
        #])) #need to check if the matrix is OK--> y.T parts
        #B = np.array([0]+[1]*len(y_values))

        #Regression
        Omega = self.kernel_(self.x, self.x)
        Ones = np.array([[1]]*len(self.y)) # needs to be a 2D 1-column vector, hence [[ ]]

        A_dag = np.linalg.pinv(np.block([
            [0,                           Ones.T                      ],
            [Ones,   Omega + self.gamma**-1 * np.identity(len(self.y))]
        ])) #need to check if the matrix is OK--> y.T parts
        B = np.concatenate((np.array([0]), self.y), axis=None)

        solution = np.dot(A_dag, B)
        self.intercept_ = solution[0]
        self.coef_      = solution[1:]


    def fit(self, X: np.ndarray, y: np.ndarray):
        """
        Fit the parameters based on the support vectors X (and store these as they are
        parameters of the LS-SVM as well, because needed for prediction)
        We are doing Regression.
        Parameters:
            - X : 2D array of vectors (1 per row: X[0,:] first vector)
            - y : 1D vector of targets
        """

        #print("IN FIT==> GAMMA=",self.gamma,"  SIGMA=",self.sigma)

        if isinstance(X, (pd.DataFrame, pd.Series)): #checks if X is an instance of either types
            Xloc = X.to_numpy()
        else:
            Xloc = X

        if isinstance(y, (pd.DataFrame, pd.Series)):
            yloc = y.to_numpy()
        else:
            yloc = y

        #check the dimensionality of the input
        if (Xloc.ndim == 2) and (yloc.ndim == 1):
            self.x = Xloc
            self.y = yloc
            self.__OptimizeParams()
        else:
            message = "The fit procedure requires a 2D numpy array of features "\
                "and 1D array of targets"
            raise Exception(message)

    def predict(self, X: np.ndarray)->np.ndarray:
        """
        Predict the regression values for a set of feature vectors
        Parameters:
            - X: ndarray of feature vectors (max: 2D), 1 per row if more than one.
        """
        Ker = self.kernel_(X, self.x) #second component should be the array of training vectors
        Y = np.dot(self.coef_, Ker.T) + self.intercept_
        return Y

In [ ]:
def svm2(produto):
  topprod_index = raw_data['CodProduto'] == produto
  topprod_raw = raw_data.loc[topprod_index]

  model = LSSVMRegression()
  x = Product(topprod_raw)
  x.make_returns()
  x.train_val()
  val = x.data_val
  train = x.data_train

  change_dataframe_train = train["Relative change over the next one month"]
  change_train = change_dataframe_train.to_numpy()

  lag_matrix_frame_train = train.iloc[ : , 3:]
  lag_matrix_train = lag_matrix_frame_train.to_numpy()

  change_dataframe_test = val["Relative change over the next one month"]
  change_test = change_dataframe_test.to_numpy()

  lag_matrix_frame_test = val.iloc[ : , 3:]
  lag_matrix_test = lag_matrix_frame_test.to_numpy()

  model.fit(lag_matrix_train, change_train)

  np.set_printoptions(suppress=True)
  predict = model.predict(lag_matrix_test)
  error = predict - change_test

  # Finding predictions with real values
  dftemp = pd.DataFrame({'predict': predict})
  dftemp = dftemp.reset_index()

  tamanho_total = 83
  real_value_test = pd.DataFrame({'real_values' : topprod_raw['QuantdadeItem'].iloc[tamanho_total-len(change_test):]})
  #print(real_value_test)
  real_value_test = real_value_test.reset_index()
  #print(real_value_test)
  predict_df = pd.DataFrame({'predict': predict})
  #print(predict_df)

  #print(real_value_test)
  predict_value = (real_value_test['real_values'])*(predict_df['predict'].shift(-1)+1)
  return predict, error, predict_value

In [ ]:
def preve_vendas_gambiarra_horizon(dataframe_novo, produto, temporal, horizon):
  #temporal '3M' = Trimestral, '6W' = 45 em 45 dias

  df = dataframe_novo
  prod = produto
  temp = temporal
  hor = horizon

  df = df.reset_index()

  base_do_df_gambiarra = {'DtEmissao': df['DtEmissao'], 'QuantdadeItem': (df[prod].astype('float')).replace(0.0, 0.1)}

  time_series = pd.DataFrame(base_do_df_gambiarra)
  #print('time_series')
  #print('head')
  #print(time_series.head(35))
  #print('tail')
  #print(time_series.tail(35))
  #print('info')
  #print(time_series.info())

  #print(time_series[time_series.applymap(lambda x: x is None).any(axis=1)])

  #aux_series = time_series["QuantdadeItem"].shift(1)
  #change = time_series['QuantdadeItem'].iloc[1:].div(aux_series) - 1
  change = time_series['QuantdadeItem'].pct_change()
  #print(change)

  data = {'DtEmissao':time_series['DtEmissao'],'QuantdadeItem':time_series['QuantdadeItem'],
                'Relative change over the next one month':change.shift(-1),
                'Lag 0 months':change}
  # i é contador de produtos
  i = 0
  #flag é indicador:
  # flag = 0 é produto com inf no modelo (crash)
  # flag = 1 é produto com len < 14 (crash)
  # flag = 2 é produto que roda adequadamente
  flag = np.nan

  while i <= lagnumber-1:
    data.update({'Lag '+str(i)+' months':change.shift(i) })
    i = i + 1
  data = pd.DataFrame(data=data)

  tamanho = len(data['DtEmissao'])

  # Usa de lagnumber até tamanho-1. Mudar este trecho quando formos fazer o api para incluir
  # a quantidade que acharmos necessária
  df_change = data.iloc[lagnumber:tamanho-1 , 2:3]
  df_lagmatrix = data.iloc[lagnumber:tamanho-1, 3:]
  change = df_change['Relative change over the next one month']
  lagmatrix = df_lagmatrix.to_numpy()

  #checam se há inf em algum lugar
  inf_em_change = (np.isinf(df_change).any()).sum()
  inf_em_lagmatrix = (np.isinf(df_lagmatrix).any()).sum()

  #print(produto)
  #print('inf em change', inf_em_change)
  #print('inf em lagmatrix', inf_em_lagmatrix)
  #print('soma', inf_em_change +  inf_em_lagmatrix)
  predict = np.inf
  lista_predicao = []

  if (inf_em_change + inf_em_lagmatrix)>0:
    #predict = np.inf
    flag = 0
  #print(df_change.columns.to_series()[np.isinf(df_change).any()])
  #print(df_lagmatrix.columns.to_series()[np.isinf(df_lagmatrix).any()])
    print(produto)
    print(time_series['QuantdadeItem'].head(35))
    print(time_series['QuantdadeItem'].tail(35))
    print(change)
    print(change.head(35))
    print(change.tail(35))

  elif len(data)<14:
    #predict = np.nan
    flag = 1
    print(produto)
    print('flag', flag)


  else:
    df_lagmatrix_saida = data.iloc[tamanho-1:tamanho, 3:]
    lagmatrix_saida = df_lagmatrix_saida.to_numpy()
    #print(produto)
    #print('lagmatrix')
    #print(lagmatrix)
    #print('lagmatrix_saida')
    #print(lagmatrix_saida)


    model = LSSVMRegression()
    #print('model')
    #print('data')
    #print(data)
    #print('lagmatrix')
    #print(lagmatrix)
    #print('change')
    #print(change)

    #print('len lagmatrix', len(lagmatrix))
    #print('len data', len(data))
    #print('len change', len(change))

    model.fit(lagmatrix, change)
    #print('model.fit')
    predict = model.predict(lagmatrix_saida)[0]
    #print(predict[0])

    lista_predicao.insert(0, predict)

    for i in list(range(hor-1)):
        #print('i', i)
        aux = lista_predicao[i]
        #print('aux', aux)
        lagmatrix_saida = lagmatrix_saida.tolist()
        #print('lagmatrix_saida', lagmatrix_saida)
        #print('lagmatrix_saida[0]', lagmatrix_saida[0])
        lagmatrix_saida = np.array([fila_anda(lagmatrix_saida[0], aux)])
        #print('lagmatrix_saida depois', lagmatrix_saida)
        predict = model.predict(lagmatrix_saida)[0]
        #print('predict', predict)
        lista_predicao.insert(i+1, predict)
        #print('lista predicao', lista_predicao)

  return lista_predicao, flag, time_series['QuantdadeItem'].iloc[-1]

In [ ]:
def cria_df_produtos_previsao_gambiarra_horizon(dataframe, dataframe_produtos, temporal, horizon):

    df = dataframe
    df_produtos = dataframe_produtos
    temp = temporal
    hor = horizon

    data = df.columns[1:]
    #print('colunas do primeiro', data)
    total = len(data)
    #print('len', total)
    dfaux = pd.DataFrame(data, columns=['CodProduto'])

    #print(dfaux)
    i = 0
    # Contadores de produtos com erros inf e len
    count_inf = []
    count_len = []
    lista_preve_var = []
    lista_preve_quant = []

    for produto in data:

        #print('produto', produto)
        previsao = preve_vendas_gambiarra_horizon(df, produto, temp, hor)
        #print('previsao', previsao)
        #dfaux[dfaux['CodProduto']==produto]['previsao '+temporal]=previsao[0]
        #print('previsao', previsao)
        #print('previsao[0]', previsao[0])
        lista_preve_var.append(previsao[0])
        #print('lista_preve_var', lista_preve_var)


        aux = previsao[2]*(1+previsao[0][0])
        lista_preve_quant_produto = [aux]
        #print('lista_preve_quant_produto')
        #print(lista_preve_quant_produto)

        for j in list(range(hor-1)):
            #print('i', i)
            aux = aux * (1 + previsao[0][j+1])
            #print(aux)
            lista_preve_quant_produto.append(aux)
            #print('lista_preve_quant_produto')
            #print(lista_preve_quant_produto)

        lista_preve_quant.append(lista_preve_quant_produto)
        #print('lista_preve_quant', lista_preve_quant)
        for k in list(range(hor)):
            dfaux.loc[dfaux['CodProduto'] == produto, 'previsao quant '+temporal+' h'+ str(k+1)] = lista_preve_quant_produto[k]

        for k in list(range(hor)):
            dfaux.loc[dfaux['CodProduto'] == produto, 'previsao var '+temporal+' h'+ str(k+1)] = previsao[0][k]
        dfaux.loc[dfaux['CodProduto'] == produto, 'ultimo da série '+temporal] = previsao[2]
        #print('previsao[0]', previsao[0])
        #print(dfaux[dfaux['CodProduto']==produto])


    if previsao[1]==0:
        count_inf.append(i)
        #print('count_inf', count_inf)
        print('inf i', i, '/', total)
    elif previsao[1]==1:
        count_len.append(i)
        #print('count_len', count_len)
        print('len i', i, '/', total)
    else:
        #print('norm i', i, '/', total)
        ;
        i = i + 1

    #print(dfaux)
    #dfaux['previsao vendas var '+temp] = lista_preve_var
    #dfaux['previsao vendas quant '+temp] = lista_preve_quant

    return dfaux, count_inf, count_len

In [ ]:
def fila_anda(lista, inserir_inicio):
    lista.insert(0, inserir_inicio)
    lista.pop()
    return lista

# **Filtro por Curvas**

In [ ]:
# Reloading
with open(file = "df_adalbertoA.csv", mode = 'r') as f:
    df_adalbertoA = pd.read_csv(f, index_col=0)
with open(file = "df_adalbertoB.csv", mode = 'r') as f:
    df_adalbertoB = pd.read_csv(f, index_col=0)
with open(file = "df_adalbertoC.csv", mode = 'r') as f:
    df_adalbertoC = pd.read_csv(f, index_col=0)

In [ ]:
list_adalbertoA = list(df_adalbertoA.CodProduto.unique())
list_adalbertoB = list(df_adalbertoB.CodProduto.unique())
list_adalbertoC = list(df_adalbertoC.CodProduto.unique())

# **Atualizações**

In [ ]:
# ATUALIZACAO MENSAL
#OBS - Data do mês que vai iniciar a atualização.
#Exemplo: 1/10/2023 vai concatenar o histórico concatenado do mês 08 e o linha do ultimo dia do mês 09
data = datetime.date(2023, 11, 1)
data_penultimo_fim = data - datetime.timedelta(days=1)
data_antepenultimo_inicio = data_penultimo_fim.replace(day=1)
data_antepenultimo_fim = data_antepenultimo_inicio - datetime.timedelta(days=1)
#hora = '0830'
#data_inicio = data.replace(day=1)

#SET UP
dia = data.day
mes = data.month
mes_anterior = data_penultimo_fim.month
ano = data.year
s_dia = '{:02d}'.format(data.day)
s_mes = '{:02d}'.format(data.month)
s_mes_penultimo = '{:02d}'.format(data_penultimo_fim.month)
s_mes_antepenultimo = '{:02d}'.format(data_penultimo_fim.month-1)
s_dia_penultimo_fim = '{:02d}'.format(data_penultimo_fim.day)
s_dia_antepenultimo_fim = '{:02d}'.format(data_antepenultimo_fim.day)
s_ano = str(ano)
s_data = s_ano+ s_mes + s_dia
s_data_penultimo_fim = s_ano + s_mes_penultimo + s_dia_penultimo_fim
s_data_antepenultimo_fim = s_ano + s_mes_antepenultimo + s_dia_antepenultimo_fim
#s_dia_primeiro = '{:02d}'.format(data_inicio.day)
#s_data_inicio = s_ano + s_mes + s_dia_primeiro


# Reloading
with open(file = "df_hist_"+ s_data_antepenultimo_fim +"_tot1M.csv", mode = 'r') as f:
    df_hist_anterior_concatenado = pd.read_csv(f, index_col=0)
with open(file = "df_hist_" +s_data_penultimo_fim + "_linha.csv", mode = 'r') as f:
    df_hist_atual_M = pd.read_csv(f, index_col=0)


atual_total = atualizar2(df_hist_anterior_concatenado, df_hist_atual_M)
print('Terminou concatenação!')

df_hist_atual_concatenado_totM = atual_total[0]
df_hist_atual_concatenado_tot2M = atual_total[1]
df_hist_atual_concatenado_tot3M = atual_total[2]

# Salvando
df_hist_atual_concatenado_totM.to_csv(r'df_hist_'+ s_data_penultimo_fim +'_tot1M.csv')

lista_ABC = list_adalbertoA + list_adalbertoB + list_adalbertoC
df_hist_atual_concatenado_totMABC = df_hist_atual_concatenado_totM.loc[lista_ABC]
df_hist_atual_concatenado_tot2MABC = df_hist_atual_concatenado_tot2M.loc[lista_ABC]
df_hist_atual_concatenado_tot3MABC = df_hist_atual_concatenado_tot3M.loc[lista_ABC]

# Salvando
df_hist_atual_concatenado_totMABC.to_csv(r'df_hist_'+ s_data_penultimo_fim +'_tot1MABC.csv')
df_hist_atual_concatenado_tot2MABC.to_csv(r'df_hist_'+ s_data_penultimo_fim +'_tot2MABC.csv')
df_hist_atual_concatenado_tot3MABC.to_csv(r'df_hist_'+ s_data_penultimo_fim +'_tot3MABC.csv')

df_hist_atual_concatenado_totM01ABC = df_hist_atual_concatenado_totMABC.replace(0.0, 0.1)
df_hist_atual_concatenado_tot2M01ABC = df_hist_atual_concatenado_tot2MABC.replace(0.0, 0.1)
df_hist_atual_concatenado_tot3M01ABC = df_hist_atual_concatenado_tot3MABC.replace(0.0, 0.1)

df1 = df_hist_atual_concatenado_totM01ABC
df1 = df1.transpose()
df1 = df1.reset_index()
df1.columns = ['DtEmissao'] + list(df1.columns[1:])
df_hist_atual_concatenado_totM01ABC_t = df1

df2 = df_hist_atual_concatenado_tot2M01ABC
df2 = df2.transpose()
df2 = df2.reset_index()
df2.columns = ['DtEmissao'] + list(df2.columns[1:])
df_hist_atual_concatenado_tot2M01ABC_t = df2

df3 = df_hist_atual_concatenado_tot3M01ABC
df3 = df3.transpose()
df3 = df3.reset_index()
df3.columns = ['DtEmissao'] + list(df3.columns[1:])
df_hist_atual_concatenado_tot3M01ABC_t = df3

import time

print('Começou o Modelo')
start = time.time()

temporal = 'M'
horizon = 3
df = df_hist_atual_concatenado_totM01ABC_t
preve_total_testeM = cria_df_produtos_previsao_gambiarra_horizon(df, df, temporal, horizon)
df_atual_tot_previsaoMABC = preve_total_testeM[0]

end = time.time()
tot1 = (end - start)/60
print('1', tot1)

start = time.time()

temporal = '2M'
horizon = 3
df = df_hist_atual_concatenado_tot2M01ABC_t
preve_total_teste2M = cria_df_produtos_previsao_gambiarra_horizon(df, df, temporal, horizon)
df_atual_tot_previsao2MABC = preve_total_teste2M[0]

end = time.time()
tot2 = (end - start)/60
print('2', tot2)

start = time.time()

temporal = '3M'
horizon = 3
df = df_hist_atual_concatenado_tot3M01ABC_t
preve_total_teste3M = cria_df_produtos_previsao_gambiarra_horizon(df, df, temporal, horizon)
df_atual_tot_previsao3MABC = preve_total_teste3M[0]

end = time.time()
tot3 = (end - start)/60
print('3', tot3)
total = tot1 + tot2 + tot3
print('total' , total)

# Salvando
df_atual_tot_previsaoMABC.to_csv(r'df_'+ s_data_penultimo_fim +'_tot_svm1MABC.csv')
df_atual_tot_previsao2MABC.to_csv(r'df_'+ s_data_penultimo_fim +'_tot_svm2MABC.csv')
df_atual_tot_previsao3MABC.to_csv(r'df_'+ s_data_penultimo_fim +'_tot_svm3MABC.csv')

# Linear

df_copia_linear = df_hist_atual_concatenado_totMABC.transpose()
df_copia_linear = df_copia_linear.reset_index()
df_copia_linear.columns = ['DtEmissao'] + list(df_copia_linear.columns[1:])
#df_copia_linear.DtEmissao.iloc[-1] = df_copia_linear.DtEmissao.iloc[-1] + MonthEnd(1)
prepre = linear_prediction(df_copia_linear, 1)
testao = [prepre[i][0] for i, elemento in enumerate(prepre)]
testao = [round(elemento) for elemento in testao]

df_testarada = df_copia_linear
df_testarada = df_testarada.set_index('DtEmissao')
df = df_testarada
df.loc[len(df.index)] = testao

df = df.transpose()

# Salvando
df.to_csv(r'df_'+ s_data_penultimo_fim +'_tot_linearMABC.csv')